# BusNet on Sort-of-CLEVR — quick test

Six object modules and one **head** that holds only the question, all on
**one bus**: every module writes $m_j e^{i\theta_j}$, every receiver gets
the sum and demodulates with its own phase,
$r_i = \sum_j m_j \cos(\theta_j - \theta_i)$. The head hears nothing else, so
the answer can only be assembled from what reaches it over the wire.

On the standard task a binary question needs the head to hear five objects
over one wire, which one phase cannot separate at once, so the model has
to multiplex in time (senders at different natural frequencies drifting
through the head's phase, or the head sweeping) or the senders must take
turns. `head_tvar` and the phase-trajectory figure are where that shows.

| arm | question |
|---|---|
| `bus/phase` | the model |
| `bus/open` | every phase 0: the head gets the plain sum (interference) |
| `bus/zero` | silent bus: the floor (the head knows only the question) |
| `channels/attn` | per-sender access restored: the ceiling |
| `bus/phase stim` | modules set their own phase from their state |
| `bus/phase omega` | fixed frequency spread: time-division by drift |
| `bus/phase d3` / `d6` | phases on $S^{d-1}$ read through a receiver-centred frame: $d$ channels at once |

Read `eval/test model/head_own_align` vs `head_other_align`, `head_tvar`,
`test_interventions/phase_zero_drop`, and ternary accuracy above .538.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys, os
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'main.py').exists())
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)

import torch
import torch.nn as nn

from src.core.config import Config, TrainConfig, LoggingConfig, OptimConfig
from src.tasks.sort_of_clevr.config import SortOfClevrDataConfig
from src.tasks.sort_of_clevr.callbacks import AccuracyCallbackCfg, QtypeAccuracyCallbackCfg
from src.tasks.sort_of_clevr.callbacks.interventions import InterventionsCallbackCfg
from src.tasks.sort_of_clevr.callbacks.binding_analysis import BindingAnalysisCfg
from src.tasks.sort_of_clevr.callbacks.t_variance import TVarianceCallbackCfg
from src.tasks.sort_of_clevr.models.busnet import SortOfClevrBusNet, SortOfClevrBusNetConfig


In [ ]:
from accelerate import Accelerator
from accelerate.utils import ProjectConfiguration

from src import build_dataloaders, build_optim, build_lr_scheduler, build_loss_fn, build_callbacks
from src.training import Trainer
from src.training.utils import set_seed


def train(model: nn.Module, cfg: Config, out_dir: str):
    """Train and return (test metrics, trainer). The trainer carries
    `intervention_results` and `binding_results` set by the callbacks."""
    os.makedirs(out_dir, exist_ok=True)
    set_seed(cfg.train.seed)
    accelerator = Accelerator(
        mixed_precision=cfg.train.mixed_precision,
        gradient_accumulation_steps=cfg.train.grad_accum,
        project_config=ProjectConfiguration(project_dir=out_dir),
    )
    dataloaders = build_dataloaders(cfg, str(accelerator.device))
    optimiser = build_optim(model, cfg.optim)
    scheduler = build_lr_scheduler(optimiser, cfg.train.n_steps, cfg.optim)
    try:
        trainer = Trainer(
            cfg=cfg, out_dir=out_dir, logger=None, model=model, dataloaders=dataloaders,
            optimiser=optimiser, scheduler=scheduler, accelerator=accelerator,
            callbacks=build_callbacks(cfg), loss_fn=build_loss_fn(cfg),
        )
        results = trainer.train()
        return results, trainer
    finally:
        accelerator.end_training()


In [ ]:
OUT_DIR = ROOT / 'notebooks' / 'outputs' / 'busnet_soc_dev'

cfg = Config(
    train=TrainConfig(
        seed=0,
        n_steps=3_000,                # quick test; the screen used 100k
        train_bs=256, val_bs=1024,
        early_stop_metric='loss', early_stop_big_is_better=False,
        early_stop_patience=1_000_000, early_stop_min_delta=0.0,
        mixed_precision='bf16', compile_model=False,
        grad_accum=1, grad_clip=1.0, loader_mode='gpu_cached', num_workers=0,
    ),
    logging=LoggingConfig(
        eval_log_interval=500, train_log_interval=100,
        info_metrics=['loss', 'accuracy', 'binary_accuracy', 'ternary_accuracy'],
        save_best=False,
    ),
    optim=OptimConfig(
        optimiser='adamw', lr=3e-4, weight_decay=0.01,
        lr_scheduler='warmup_cosine', lr_scheduler_params={'warmup_steps': 300},
    ),
    dataset=SortOfClevrDataConfig(
        name='sort_of_clevr', seed=1,
        root=r'/home/nik/workspace/ImperialWork/msc_project/SyncNetProject/data',
        dir='sort-of-clevr-notebook-dev',
        train_size=20_000, test_size=1000, img_size=75, obj_size=5, nb_questions=10, t_subtype=-1,
    ),
    callbacks=[
        AccuracyCallbackCfg(),
        QtypeAccuracyCallbackCfg(),
        InterventionsCallbackCfg(max_batches=8),
        TVarianceCallbackCfg(t_values=[0, 1, 2, 4, 8], n_repeats=1, max_batches=4),
        BindingAnalysisCfg(max_batches=2, n_examples=3),
    ],
)


In [ ]:
# prepare the dataset once if the dev split does not exist yet
# from src.tasks import TASKS
# TASKS['sort_of_clevr'].prepare(cfg.dataset)


In [ ]:
ARMS = {
    'bus/phase':       dict(),
    'bus/open':        dict(bus_phase='open'),
    'bus/zero':        dict(bus_phase='zero'),
    'channels/attn':   dict(medium='channels', gate_mode='attn'),
    'bus/phase stim':  dict(drive='stimulus'),
    'bus/phase omega': dict(learn_omega=False, omega_init=2.0),     # circle time-division
    'bus/phase d3':    dict(phase_repr='vector', osc_dim=3),          # S^2: three channels at the head
    'bus/phase d6':    dict(phase_repr='vector', osc_dim=6),          # one axis per object: private lines
    'bus/phase d6 stim': dict(phase_repr='vector', osc_dim=6, drive='stimulus'),
}

results, interventions, model_metrics = {}, {}, {}
for name, kw in ARMS.items():
    model = SortOfClevrBusNet.from_config(SortOfClevrBusNetConfig(**kw), cfg.dataset)
    print(f'\n===== {name}  ({sum(p.numel() for p in model.parameters()):,} params) =====')
    results[name], trainer = train(model, cfg, str(OUT_DIR / name.replace('/', '_')))
    interventions[name] = getattr(trainer, 'intervention_results', {})


In [ ]:
FLOORS = {'accuracy': 0.492, 'binary_accuracy': 0.433, 'ternary_accuracy': 0.538}

print(f'{"arm":<16}' + ''.join(f'{k.split("_")[0]:>10}' for k in FLOORS)
      + f'{"p_zero":>9}{"p_shuf":>9}{"p_frz":>9}{"h_own":>8}{"h_oth":>8}{"h_tvar":>8}{"R":>6}')
for name in ARMS:
    r, iv = results[name], interventions[name]
    row = ''.join(f'{r[f"callbacks/{k}"] - v:>+10.3f}' for k, v in FLOORS.items())
    row += ''.join(f'{iv.get(k, float("nan")):>+9.3f}' for k in ['phase_zero_drop', 'phase_shuffle_drop', 'phase_freeze_drop'])
    row += ''.join(f'{r.get(f"model/{k}", float("nan")):>8.2f}' for k in ['head_own_align', 'head_other_align', 'head_tvar'])
    row += f'{r.get("model/phase_R", float("nan")):>6.2f}'
    print(f'{name:<16}' + row)

print('\naccuracies are deltas above the question-only floors; drops are baseline minus intervened;')
print('h_own / h_oth = cos alignment of the head with the queried object(s) / the rest; h_tvar = how much')
print('the head\'s alignment pattern changes across steps (time-multiplexing). figures under', OUT_DIR / '<arm>' / 'viz')


**How to read it.** `bus/zero` is the floor (a head that knows only the
question), `channels/attn` the ceiling. The result is positive if
`bus/phase` sits clearly above `bus/open`, `phase_zero_drop` is positive
(setting every phase to zero at test time costs what the open bus costs),
and `h_own` exceeds `h_oth`. A high `h_tvar` with good accuracy is the
time-multiplexing signature: the head is visiting senders on different
steps. If `bus/phase` ends at `R` near 1, the phases have collapsed to
global synchrony and the bus is an open bus.
